# ERP: encoding or retrieval vs. baseline

Band-pass filtered signal after encoding or retrieval onset, compared with the fixation-cross baseline.
Set `PHASE` to `"encoding"` or `"retrieval"` and `BAND` to `"theta"`, `"low gamma"` or `"high gamma"`.
The band limits are defined in `BANDS` in `helpers.py`.

In [ ]:
from pathlib import Path
import numpy as np
from helpers import (BANDS, load_data, bandpass_filter, find_onsets, extract_epochs,
                     compare_to_baseline, fdr, significant_windows, plot_erp)

PHASE = "retrieval"              # "encoding" or "retrieval"
BAND = "theta"                   # "theta", "low gamma" or "high gamma"

DATA_DIR = "data"
PATIENT = "kh9_1"
SR = 1024                        # sampling rate (Hz)
WINDOW = (-0.5, 0.5)             # s around event onset
ALPHA = 0.05
MIN_SIG_MS = 5                   # shortest significant window that is marked in the plot

# channels averaged together in the plots
GROUPS = {"LB": ["LB8", "LB9"], "LK": ["LK5", "LK7"]}
COLORS = {"LB": "#32CD32", "LK": "#008B8B"}

OUT_DIR = Path("results") / PHASE / BAND.replace(" ", "_") / "erp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and filter

In [ ]:
data, events, channel_names = load_data(DATA_DIR, PATIENT)
filtered = bandpass_filter(data, *BANDS[BAND], SR)
print(data.shape, BAND, BANDS[BAND], "Hz")

## 2. Trials

Event trials: 0.5 s before to 0.5 s after each encoding or retrieval onset.
Baseline: 1 s starting at each fixation cross.
Shape: (trials, time points, channels).

In [ ]:
start = int(WINDOW[0] * SR)
length = int((WINDOW[1] - WINDOW[0]) * SR)

trials = extract_epochs(filtered, find_onsets(events, PHASE.capitalize()), start, length)
baseline = extract_epochs(filtered, find_onsets(events, "FixationCross", exact=True), 0, length)
print(PHASE, trials.shape)
print("baseline", baseline.shape)

## 3. Statistics

Mann-Whitney U test at every time point and channel, FDR-corrected across time points within each channel.

In [ ]:
p = compare_to_baseline(trials, baseline)      # (time points, channels)
sig = fdr(p, axis=0) < ALPHA

for ch in np.where(sig.any(axis=0))[0]:
    print(f"{channel_names[ch]}: {sig[:, ch].sum()} significant time points")

## 4. Plots

Average of the channels in each group (only channels with significant time points are included).
Shaded: time points that are significant in at least one of these channels.

In [ ]:
time_ms = (np.arange(length) + start) / SR * 1000

for group, names in GROUPS.items():
    idx = [np.where(channel_names == name)[0][0] for name in names]
    idx = [ch for ch in idx if sig[:, ch].any()]
    if not idx:
        print(f"{group}: no significant channels")
        continue

    windows = significant_windows(sig[:, idx].any(axis=1), time_ms, MIN_SIG_MS)
    plot_erp(time_ms, trials[:, :, idx].mean(axis=2), baseline[:, :, idx].mean(axis=2),
             windows, label=PHASE.capitalize(), color=COLORS[group],
             title=f"{group} – {PHASE}, {BAND}", save_path=OUT_DIR / f"{group}.png")